<a href="https://colab.research.google.com/github/Asheesh1272/Asheesh127/blob/main/RAG_with_open_source_multi_modal3.ipynb" target="_parent"><img src="https://colab.research.google.com/assets/colab-badge.svg" alt="Open In Colab"/></a>

In [ ]:
!pip install -q timm einops wikipedia chromadb open_clip_torch


In [ ]:
!pip install -q transformers==4.36.0
!pip install -q bitsandbytes==0.41.3 accelerate==0.25.0

In [ ]:
import kagglehub

# Download latest version
path = kagglehub.dataset_download("imsparsh/flowers-dataset")

print("Path to dataset files:", path)

In [ ]:
import wikipedia
from tqdm import tqdm
import shutil
import os


images_pth = path # Use the extracted dataset path

new_pth = 'flowers'
if not os.path.exists(new_pth):
    os.mkdir(new_pth)

# Get all items in the root dataset path and filter for directories (e.g., 'train', 'test')
main_subdirs = [d for d in os.listdir(images_pth) if os.path.isdir(os.path.join(images_pth, d))]

for split_dir_name in tqdm(main_subdirs):
    split_dir_path = os.path.join(images_pth, split_dir_name)
    new_split_dir_path = os.path.join(new_pth, split_dir_name)

    if not os.path.exists(new_split_dir_path):
        os.mkdir(new_split_dir_path)

    # Now iterate through the actual flower classes inside each split directory
    flower_classes = [c for c in os.listdir(split_dir_path) if os.path.isdir(os.path.join(split_dir_path, c))]

    for class_name in flower_classes: # Iterate through 'sunflower', 'tulip', etc.
        class_path = os.path.join(split_dir_path, class_name)
        new_class_path = os.path.join(new_split_dir_path, class_name)

        if not os.path.exists(new_class_path):
            os.mkdir(new_class_path)

        # Copy up to 10 images from each class folder
        for img_name in os.listdir(class_path)[:10]:
            img_src_path = os.path.join(class_path, img_name)
            img_dest_path = os.path.join(new_class_path, img_name)
            shutil.copy(img_src_path, img_dest_path)

In [ ]:
imges_classes


In [ ]:
# secondly we will get text from wiki and save it in txt file
wiki_titles = { # the key is imgs class and the value is wiki title
    'daisy': 'Bellis perennis',
    'dandelion': 'Taraxacum',
    'lotus': 'Nelumbo nucifera',
    'rose': 'Rose',
    'sunflower': 'Common sunflower',
    'tulip': 'Tulip',
    'bellflower':'Campanula'
}

In [ ]:
# each class has 10 images and one text file content from the wiki page

# The 'new_pth' variable and 'wiki_titles' dictionary are available from previous cells.

# Collect all unique flower class names that were copied to the 'flowers' directory
# and are present in the wiki_titles dictionary.
processed_flower_classes = set()
for split_dir_name in ['train', 'test']:
    split_dir_path = os.path.join(new_pth, split_dir_name)
    if os.path.exists(split_dir_path):
        for class_name in os.listdir(split_dir_path):
            if os.path.isdir(os.path.join(split_dir_path, class_name)) and class_name in wiki_titles:
                processed_flower_classes.add(class_name)

# Now iterate over these unique flower class names to fetch and save Wikipedia content
for cls_name in tqdm(list(processed_flower_classes)):
    # Define the path where the text file will be saved.
    # For consistency, we'll save it into a dedicated folder for each class within 'flowers',
    # e.g., 'flowers/daisy/daisy.txt'.
    text_output_dir = os.path.join(new_pth, cls_name)

    # Ensure the directory for this class exists before saving the text file
    if not os.path.exists(text_output_dir):
        os.makedirs(text_output_dir) # Use os.makedirs to create the directory if it doesn't exist

    page_content = wikipedia.summary(wiki_titles[cls_name], auto_suggest=False)

    # Save the text file
    file_name = cls_name + '.txt'
    with open(os.path.join(text_output_dir, file_name), 'w') as f:
        f.write(page_content)

In [ ]:
!mkdir all_data # this folder will contain the in very messy way
! cp -r flowers/*/* all_data

In [ ]:
# Remove the previous `all_data` directory if it exists to ensure a clean copy
import shutil
if os.path.exists('all_data'):
    shutil.rmtree('all_data')

# Create the all_data directory
os.makedirs('all_data', exist_ok=True)

# Copy all contents from the 'flowers' directory into 'all_data'
# The 'flowers' directory is located directly in the current working directory (/content/)
!cp -r flowers/*/* all_data

In [ ]:
import chromadb

from chromadb.utils.embedding_functions import OpenCLIPEmbeddingFunction
from chromadb.utils.data_loaders import ImageLoader
from chromadb.config import Settings


client = chromadb.PersistentClient(path="DB")

embedding_function = OpenCLIPEmbeddingFunction()
image_loader = ImageLoader()

In [ ]:
collection_images = client.create_collection(
    name='multimodal_collection_images',
    embedding_function=embedding_function,
    data_loader=image_loader)

collection_text = client.create_collection(
    name='multimodal_collection_text',
    embedding_function=embedding_function,
    )


In [ ]:
# Get the uris to the images
IMAGE_FOLDER = 'all_data'

image_uris = []
for root, _, files in os.walk(IMAGE_FOLDER):
    for file in files:
        file_path = os.path.join(root, file)
        # Check if it's an image file by common extensions
        if file.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp', '.tiff')):
            image_uris.append(file_path)

image_uris.sort() # Ensure consistent ordering for ids
ids = [str(i) for i in range(len(image_uris))]

collection_images.add(ids=ids, uris=image_uris)

In [ ]:
# let's test and play with our VEC-DB


In [ ]:
from matplotlib import pyplot as plt

retrieved = collection_images.query(query_texts=["tulip"], include=['data'], n_results=3)
for img in retrieved['data'][0]:
    plt.imshow(img)
    plt.axis("off")
    plt.show()

In [ ]:
from matplotlib import pyplot as plt

retrieved = collection_images.query(query_texts=["bellflower"], include=['data'], n_results=3)
for img in retrieved['data'][0]:
    plt.imshow(img)
    plt.axis("off")
    plt.show()

In [ ]:
from PIL import Image
import numpy as np
import os
from matplotlib import pyplot as plt

# Dynamically find a daisy image path from the 'all_data' directory
daisy_folder = os.path.join('all_data', 'daisy')
query_image_path = None

if os.path.exists(daisy_folder):
    daisy_images = [f for f in os.listdir(daisy_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp', '.tiff'))]
    if daisy_images:
        # Pick the first daisy image found
        query_image_path = os.path.join(daisy_folder, daisy_images[0])

if query_image_path:
    query_image = np.array(Image.open(query_image_path))
    print("Query Image")
    plt.imshow(query_image)
    plt.axis('off')
    plt.show()

    print("Results")
    retrieved = collection_images.query(query_images=[query_image], include=['data'], n_results=3)
    for img in retrieved['data'][0][1:]:
        plt.imshow(img)
        plt.axis("off")
        plt.show()
else:
    print("Could not find a daisy image in 'all_data/daisy' to use as query. Please ensure the 'daisy' folder exists and contains images.")

In [ ]:
from PIL import Image
import numpy as np
import os
from matplotlib import pyplot as plt

# Dynamically find a rose image path from the 'all_data' directory
rose_folder = os.path.join('all_data', 'rose')
query_image_path = None

if os.path.exists(rose_folder):
    rose_images = [f for f in os.listdir(rose_folder) if f.lower().endswith(('.png', '.jpg', '.jpeg', '.gif', '.bmp', '.tiff'))]
    if rose_images:
        # Pick the first rose image found
        query_image_path = os.path.join(rose_folder, rose_images[0])

if query_image_path:
    query_image = np.array(Image.open(query_image_path))
    print("Query Image")
    plt.imshow(query_image)
    plt.axis('off')
    plt.show()

    print("Results")
    retrieved = collection_images.query(query_images=[query_image], include=['data'], n_results=3)
    for img in retrieved['data'][0][1:]:
        plt.imshow(img)
        plt.axis("off")
        plt.show()
else:
    print("Could not find a rose image in 'all_data/rose' to use as query. Please ensure the 'rose' folder exists and contains images.")

In [ ]:
# now the text DB
from chromadb.utils import embedding_functions
default_ef = embedding_functions.DefaultEmbeddingFunction()

text_pth = sorted([os.path.join(IMAGE_FOLDER, image_name) for image_name in os.listdir(IMAGE_FOLDER) if image_name.endswith('.txt')])

In [ ]:
text_pth

In [ ]:
list_of_text = []
for text in text_pth:
    with open(text, 'r') as f:
        text = f.read()
        list_of_text.append(text)

In [ ]:
ids_txt_list = ['id'+str(i) for i in range(len(list_of_text))]
ids_txt_list

In [ ]:
collection_text.add(
    documents = list_of_text,
    ids =ids_txt_list
)

In [ ]:
collection_text.add(
    documents = list_of_text,
    ids =ids_txt_list
)

In [ ]:
results = collection_text.query(
    query_texts=["What is the bellflower?"],
    n_results=1
)

results

In [ ]:
collection_text.count()

In [ ]:
# db = Chroma(persist_directory="/kaggle/working/DB\\",embedding_function=embedding)

In [ ]:
from huggingface_hub import hf_hub_download

hf_hub_download(repo_id="visheratin/LLaVA-3b", filename="configuration_llava.py", local_dir="./", force_download=True)
hf_hub_download(repo_id="visheratin/LLaVA-3b", filename="configuration_phi.py", local_dir="./", force_download=True)
hf_hub_download(repo_id="visheratin/LLaVA-3b", filename="modeling_llava.py", local_dir="./", force_download=True)
hf_hub_download(repo_id="visheratin/LLaVA-3b", filename="modeling_phi.py", local_dir="./", force_download=True)
hf_hub_download(repo_id="visheratin/LLaVA-3b", filename="processing_llava.py", local_dir="./", force_download=True)